# Zero-shot and few-shot prompting

Part of the M.S. thesis *Hate Speech Classification in Roman Urdu: A Comparative Study on Parameter Efficient Fine-Tuning and Prompt Engineering*.

Classifies Roman Urdu comments as toxic / non-toxic with **GPT-3.5** on **Azure OpenAI**, comparing a **zero-shot** system prompt against a **few-shot** prompt built from curated labelled examples. Predictions are made at temperature 0 and evaluated with accuracy and macro precision/recall/F1.

**Setup:** install dependencies with `pip install -r requirements.txt` from the repository root and place the dataset in `data/` (see the main README).

In [ ]:
import os

# Paths are relative to this notebook. See data/README.md for the dataset.
DATA_PATH = "../../data/PURRUTT.xlsx"
SPECIAL_EXAMPLES_PATH = "../../data/special_examples.csv"

In [ ]:

from openai import AzureOpenAI
import json
import random
import tiktoken

import pandas as pd
import numpy as np

from collections import Counter
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, accuracy_score
from tabulate import tabulate

In [ ]:
# Credentials are read from environment variables - never hard-code API keys.
# Set AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_KEY and AZURE_OPENAI_DEPLOYMENT (see .env.example).
creds = {
    "AZURE_OPENAI_ENDPOINT": os.environ["AZURE_OPENAI_ENDPOINT"],
    "AZURE_OPENAI_KEY": os.environ["AZURE_OPENAI_KEY"],
    "AZURE_OPENAI_APIVERSION": os.environ.get("AZURE_OPENAI_APIVERSION", "2024-05-01-preview"),
    "CHATGPT_MODEL": os.environ["AZURE_OPENAI_DEPLOYMENT"],
}

In [ ]:
client = AzureOpenAI(
    azure_endpoint=creds["AZURE_OPENAI_ENDPOINT"],
    api_key=creds["AZURE_OPENAI_KEY"],
    api_version=creds["AZURE_OPENAI_APIVERSION"]
)

In [ ]:
chat_model_id = creds["CHATGPT_MODEL"]

In [ ]:
def num_tokens_from_messages(messages):

    """
    Return the number of tokens used by a list of messages.
    Adapted from the Open AI cookbook token counter
    """

    encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")

    # Each message is sandwiched with <|start|>role and <|end|>
    # Hence, messages look like: <|start|>system or user or assistant{message}<|end|>

    tokens_per_message = 3 # token1:<|start|>, token2:system(or user or assistant), token3:<|end|>

    num_tokens = 0

    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))

    num_tokens += 3  # every reply is primed with <|start|>assistant<|message|>

    return num_tokens

In [ ]:
cs_reviews_df = pd.read_excel(DATA_PATH)
cs_reviews_df = cs_reviews_df.dropna()
# Ensure Toxic column is integer type
cs_reviews_df['Toxic'] = cs_reviews_df['Toxic']

In [ ]:
cs_reviews_df.info()

In [ ]:
cs_reviews_df.sample(5)

In [ ]:
cs_reviews_df.Toxic.value_counts()

In [ ]:
cs_reviews_df.shape

In [ ]:
cs_examples_df, cs_gold_examples_df = train_test_split(
    cs_reviews_df, #<- the full dataset
   test_size = 0.2, #<- 20% random sample selected for gold examples
    random_state=42,
    stratify = cs_reviews_df['Toxic'] #<- ensures that the splits are the same for every session
)

In [ ]:
(cs_examples_df.shape, cs_gold_examples_df.shape)

In [ ]:
columns_to_select = ['Comment','Toxic']

In [ ]:
training_examples = (
        cs_reviews_df.loc[:, columns_to_select].to_json(orient='records')
)

In [ ]:
user_message_template = """```{Comment_review}```"""

In [ ]:
zero_shot_system_message = """You are a hate speech detection assistant. You will be provided with a comment. Each comment will be enclosed by triple backticks.
You need to read the comment and classify it into one of the below 2 categories:
1- 1 for toxic comments (comments containing hate speech, offensive language, or harmful content).
2- 0 for non-toxic comments (comments free of hate speech and offensive language).

Classify the comment into these 2 categories only. DO NOT use any other classification. Categories like Neutral or Mixed are unacceptable.
If the comment contains elements of both toxic and non-toxic language, classify it based on the dominant tone of the comment, and this will be our FINAL CLASSIFICATION. Now this FINAL CLASSIFICATION should be the output.
These 2 categories - 1 (toxic) and 0 (non-toxic) are the only acceptable outputs.
Compulsory - The answer should only contain one digit indicating the FINAL CLASSIFICATION - either 1 or 0. DO NOT explain your answer.
I Repeat, unacceptable responses - Neutral, Mixed, or Explanatory text. Every comment must compulsorily be classified as either 1 (toxic) or 0 (non-toxic).
"""

In [ ]:
zero_shot_prompt = [{'role':'system', 'content': zero_shot_system_message}]
# Create zero shot prompt to be input ready for completion function

In [ ]:
num_tokens_from_messages(zero_shot_prompt)

In [ ]:
few_shot_system_message  = """You are a hate speech detection assistant. You will be provided with a comment. Each comment will be enclosed by triple backticks.
You need to read the comment and classify it into one of the below 2 categories:
1- 1 for toxic comments (comments containing hate speech, offensive language, or harmful content).
2- 0 for non-toxic comments (comments free of hate speech and offensive language).

Classify the comment into these 2 categories only. DO NOT use any other classification. Categories like Neutral or Mixed are unacceptable.
If the comment contains elements of both toxic and non-toxic language, classify it based on the dominant tone of the comment, and this will be our FINAL CLASSIFICATION. Now this FINAL CLASSIFICATION should be the output.
These 2 categories - 1 (toxic) and 0 (non-toxic) are the only acceptable outputs.
Compulsory - The answer should only contain one digit indicating the FINAL CLASSIFICATION - either 1 or 0. DO NOT explain your answer.
I Repeat, unacceptable responses - Neutral, Mixed, or Explanatory text. Every comment must compulsorily be classified as either 1 (toxic) or 0 (non-toxic).
"""

In [ ]:
examples = pd.read_csv(SPECIAL_EXAMPLES_PATH)

In [ ]:
examples.info()

In [ ]:
examples = (
        examples.loc[:, columns_to_select].to_json(orient='records')
)

In [ ]:
json.loads(examples)

In [ ]:
def create_prompt(system_message, examples, user_message_template):

    """
    Return a prompt message in the format expected by the Open AI API.
    Loop through the examples and parse them as user message and assistant
    message.

    Args:
        system_message (str): system message with instructions for hate speech classification
        examples (str): JSON string with list of examples
        user_message_template (str): string with a placeholder for comment

    Output:
        few_shot_prompt (List): A list of dictionaries in the Open AI prompt format
    """

    few_shot_prompt = [{'role':'system', 'content': system_message}]

    for example in json.loads(examples):
        example_review = example['Comment']
        example_sentiment = example['Toxic']

        few_shot_prompt.append(
            {
                'role': 'user',
                'content': user_message_template.format(
                    Comment_review=example_review
                )
            }
        )

        few_shot_prompt.append(
    {'role': 'assistant', 'content': str(example_sentiment)}
)


    return few_shot_prompt

In [ ]:
few_shot_prompt = create_prompt(
    few_shot_system_message,
    examples,
    user_message_template
)

In [ ]:
few_shot_prompt

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tabulate import tabulate
import json

def batch_data(data, batch_size):
    """
    Splits data into batches.

    Args:
        data (list or DataFrame): The dataset to be split into batches.
        batch_size (int): Number of rows per batch.

    Returns:
        List of batches, each containing up to `batch_size` items.
    """
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

def evaluate_prompt_in_batches(prompt, gold_examples, user_message_template, batch_size=100):
    """
    Evaluate the model on gold examples in batches and calculate overall metrics.

    Args:
        prompt (List): List of messages in Open AI prompt format.
        gold_examples (str): JSON string with list of gold examples.
        user_message_template (str): String with a placeholder for comment.
        batch_size (int): Number of examples per batch.

    Returns:
        Tuple: List of per-batch metrics and overall metrics (accuracy, precision, recall, F1 score).
    """
    all_metrics = []  # Store metrics for each batch
    all_predictions = []  # Store all predictions across batches
    all_ground_truths = []  # Store all ground truths across batches

    gold_examples_list = json.loads(gold_examples)  # Parse JSON into a list

    # Process data in batches
    for batch_number, batch in enumerate(batch_data(gold_examples_list, batch_size)):
        model_predictions, ground_truths, review_texts = [], [], []

        for example in batch:
            gold_input = example['Comment']
            user_input = [
                {
                    'role': 'user',
                    'content': user_message_template.format(Comment_review=gold_input)
                }
            ]

            try:
                response = client.chat.completions.create(
                    model=chat_model_id,
                    messages=prompt + user_input,
                    temperature=0,  # Low temperature for deterministic response
                    max_tokens=2  # Restrict the output to 2 tokens
                )

                # Extract and sanitize prediction
                raw_prediction = response.choices[0].message.content.strip()
                if raw_prediction.isdigit():
                    prediction = int(raw_prediction)
                else:
                    # Handle invalid predictions by defaulting to non-toxic (0)
                    # print(f"Invalid prediction '{raw_prediction}' encountered. Defaulting to 0.")
                    prediction = 0

                model_predictions.append(prediction)
                ground_truths.append(example['Toxic'])
                review_texts.append(gold_input)

            except Exception as e:
                # print(f"Error in batch {batch_number}, skipping example: {e}")
                continue

        # Add predictions and ground truths to overall lists
        all_predictions.extend(model_predictions)
        all_ground_truths.extend(ground_truths)

        # Ensure consistent data types
        ground_truths = [int(label) for label in ground_truths]
        model_predictions = [int(label) for label in model_predictions]

        # Calculate metrics for the current batch
        accuracy = accuracy_score(ground_truths, model_predictions)
        precision = precision_score(ground_truths, model_predictions, average="macro", zero_division=1)
        recall = recall_score(ground_truths, model_predictions, average="macro", zero_division=1)
        micro_f1 = f1_score(ground_truths, model_predictions, average="macro", zero_division=1)

        # Aggregate batch metrics
        batch_metrics = {
            "Batch Number": batch_number + 1,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": micro_f1
        }
        print(batch_metrics)
        all_metrics.append(batch_metrics)

    # Calculate final overall metrics for the entire dataset
    final_accuracy = accuracy_score(all_ground_truths, all_predictions)
    final_precision = precision_score(all_ground_truths, all_predictions, average="macro", zero_division=1)
    final_recall = recall_score(all_ground_truths, all_predictions, average="macro", zero_division=1)
    final_f1_score = f1_score(all_ground_truths, all_predictions, average="macro", zero_division=1)

    # Return batch metrics and overall metrics
    overall_metrics = {
        "Final Accuracy": final_accuracy,
        "Final Precision": final_precision,
        "Final Recall": final_recall,
        "Final F1 Score": final_f1_score
    }

    return all_metrics, overall_metrics

In [ ]:
# Evaluate in batches and calculate overall metrics
batch_results, overall_metrics = evaluate_prompt_in_batches(
    zero_shot_prompt, training_examples, user_message_template, batch_size=100
)

# Print batch-wise results
for metrics in batch_results:
    print(metrics)

# Print overall metrics
print("\nOverall Metrics:")
print(overall_metrics)

In [ ]:
# Evaluate in batches and calculate overall metrics
batch_results, overall_metrics = evaluate_prompt_in_batches(
    few_shot_prompt, training_examples, user_message_template, batch_size=100
)

# Print batch-wise results
for metrics in batch_results:
    print(metrics)

# Print overall metrics
print("\nOverall Metrics:")
print(overall_metrics)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tabulate import tabulate
import json

def evaluate_prompt(prompt, gold_examples, user_message_template):
    """
    Return evaluation metrics for predictions on gold examples.
    For each example, we make a prediction using the prompt. Gold labels and
    model predictions are aggregated into lists and compared to compute various metrics.

    Args:
        prompt (List): list of messages in the Open AI prompt format
        gold_examples (str): JSON string with list of gold examples
        user_message_template (str): string with a placeholder for comment

    Output:
        metrics (dict): Dictionary containing accuracy, precision, recall, and F1 score
    """
    model_predictions, ground_truths, review_texts = [], [], []

    for example in json.loads(gold_examples):
        gold_input = example['Comment']
        user_input = [
            {
                'role': 'user',
                'content': user_message_template.format(Comment_review=gold_input)
            }
        ]

        try:
            response = client.chat.completions.create(
                model=chat_model_id,
                messages=prompt + user_input,
                temperature=0,  # Low temperature for deterministic response
                max_tokens=2  # Restrict the output to 2 tokens
            )

            prediction = response.choices[0].message.content
            model_predictions.append(prediction.strip())  # Remove extra white spaces
            ground_truths.append(example['Toxic'])
            review_texts.append(gold_input)

        except Exception as e:
            continue
# # Ensure consistent data types (convert to strings)
# Ensure both lists contain integers
    ground_truths = [float(label) for label in ground_truths]
    model_predictions = [float(label) for label in model_predictions]

    # Calculate metrics
    accuracy = accuracy_score(ground_truths, model_predictions)
    precision = precision_score(ground_truths, model_predictions, average="micro", zero_division=1)
    recall = recall_score(ground_truths, model_predictions, average="micro", zero_division=1)
    micro_f1 = f1_score(ground_truths, model_predictions, average="micro", zero_division=1)

    # Tabulate results
    table_data = [[text, pred, truth] for text, pred, truth in zip(review_texts, model_predictions, ground_truths)]
    headers = ["Review", "Model Prediction", "Ground Truth"]
    print(tabulate(table_data, headers=headers, tablefmt="grid"))

    # Aggregate metrics
    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Micro-F1 Score": micro_f1
    }

    return metrics

In [ ]:
evaluate_prompt(zero_shot_prompt, training_examples, user_message_template)